In [0]:
from pyspark.sql import functions as F

# Path to your Databricks Unity Catalog Volume
file_path = "/Volumes/workspace/default/fraud_data/paysim.csv"

# Load CSV into a PySpark DataFrame
df = spark.read.csv(file_path, header=True, inferSchema=True)

print("Total Transactions Loaded:", df.count())
df.printSchema()

# Databricks built-in display() gives a rich interactive table!
display(df.limit(10))


Total Transactions Loaded: 6362620
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
1,TRANSFER,181.0,C1305486145,181.0,0.0,C553264065,0.0,0.0,1,0
1,CASH_OUT,181.0,C840083671,181.0,0.0,C38997010,21182.0,0.0,1,0
1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0
1,PAYMENT,7817.71,C90045638,53860.0,46042.29,M573487274,0.0,0.0,0,0
1,PAYMENT,7107.77,C154988899,183195.0,176087.23,M408069119,0.0,0.0,0,0
1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.0,0,0
1,PAYMENT,4024.36,C1265012928,2671.0,0.0,M1176932104,0.0,0.0,0,0
1,DEBIT,5337.77,C712410124,41720.0,36382.23,C195600860,41898.0,40348.79,0,0


In [0]:
from pyspark.sql import functions as F

# 1. Check Null Count for every single column
null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
print("=== NULL COUNTS PER COLUMN ===")
display(null_counts)

# 2. Check Exact Duplicate Rows
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
duplicate_count = total_rows - distinct_rows

print(f"Total Rows: {total_rows}")
print(f"Exact Duplicate Rows Found: {duplicate_count}")

# 3. Check Duplicate Transaction IDs (same step, sender, receiver, and amount)
business_dup_count = total_rows - df.dropDuplicates(["step", "nameOrig", "nameDest", "amount"]).count()
print(f"Business Key Duplicate Rows Found: {business_dup_count}")



=== NULL COUNTS PER COLUMN ===


step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,0,0,0,0,0,0,0,0,0,0


Total Rows: 6362620
Exact Duplicate Rows Found: 0
Business Key Duplicate Rows Found: 0


In [0]:
# Skip dropDuplicates() since our data profiling proved 0 duplicates exist!
df_enriched = df \
    .withColumn("balance_drop", F.col("oldbalanceOrg") - F.col("newbalanceOrig")) \
    .withColumn("is_account_emptied", F.when((F.col("oldbalanceOrg") > 0) & (F.col("newbalanceOrig") == 0), 1).otherwise(0))

# Save as an official Delta Table inside your workspace
df_enriched.write.format("delta").mode("overwrite").saveAsTable("workspace.default.clean_transactions")

print("Cleaned data saved as Delta Table: workspace.default.clean_transactions!")


Cleaned data saved as Delta Table: workspace.default.clean_transactions!


In [0]:
%sql
SELECT 
    type AS transaction_type,
    COUNT(*) AS total_txns,
    SUM(isFraud) AS fraud_count,
    ROUND(SUM(isFraud) * 100.0 / COUNT(*), 3) AS fraud_rate_pct
FROM workspace.default.clean_transactions
GROUP BY type
ORDER BY fraud_count DESC;


transaction_type,total_txns,fraud_count,fraud_rate_pct
CASH_OUT,2237500,4116,0.184
TRANSFER,532909,4097,0.769
DEBIT,41432,0,0.000
CASH_IN,1399284,0,0.000
PAYMENT,2151495,0,0.000
